# Enfoque 4 — Recomendación con TF-IDF

Cuarta estrategia de representación, basada en **coincidencia léxica** en lugar de semántica.

A diferencia de los enfoques con *Sentence-Transformer* y *Topic Modeling* (que usan embeddings densos), acá representamos cada película con un vector **TF-IDF** sobre su texto. La idea:

- **TF (term frequency):** cuántas veces aparece cada término en el documento.
- **IDF (inverse document frequency):** penaliza términos que aparecen en muchas películas (poco discriminativos) y premia los específicos.

Cada película queda como un vector disperso (*sparse*) de dimensión = tamaño del vocabulario, y recomendamos por **similitud coseno**, igual que en los otros enfoques. Esto nos da un **baseline interpretable** y un contraste directo: TF-IDF captura coincidencias de **palabras exactas** (géneros, keywords, nombres de director, años), mientras que los embeddings capturan **significado** aunque las palabras no coincidan.

## Instalaciones e imports

In [1]:
import pandas as pd
import numpy as np
import re
import html
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Carga de datasets

Usamos los mismos insumos que el resto del trabajo: el corpus de películas y los 14 perfiles de usuario.

In [2]:
BASE = "https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/data/"
df_pelis = pd.read_csv(BASE + "peliculas.csv")
usuarios = pd.read_csv(BASE + "usuarios.csv")
print("Películas:", len(df_pelis), "| Usuarios:", len(usuarios))

Películas: 4967 | Usuarios: 14


### Limpieza inicial y construcción del id

Decodificamos las entidades HTML (p. ej. `&apos;` a comilla) en las columnas de texto y agregamos un `id` único por película a partir del índice.

In [3]:
for col in df_pelis.columns:
    if df_pelis[col].dtype == 'object':
        df_pelis[col] = df_pelis[col].apply(lambda x: html.unescape(str(x)) if pd.notna(x) else x)

df_pelis["id"] = df_pelis.index + 1

## Preprocesado

### Función de limpieza de texto
Removemos HTML residual, caracteres extraños y espacios múltiples. Es **el mismo pipeline** que en los otros enfoques, para que la comparación sea justa.

In [4]:
def limpiar_texto(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)        # remover HTML si hubiera
    text = re.sub(r'[^\w\s\.,;:!?áéíóúüñ-]', ' ', text)  # caracteres extraños
    text = re.sub(r'\s+', ' ', text)             # espacios múltiples
    return text.strip()

### Construcción del texto de cada película

Unificamos en un solo string los campos con señal de contenido: **nombre, sinopsis, director, año, género y keywords**. En TF-IDF esto es especialmente útil: los géneros y keywords aportan términos que pueden coincidir *literalmente* con lo que pide la query del usuario.

In [5]:
df_pelis["texto"] = (
    df_pelis["name"].apply(limpiar_texto) + ". "
    + df_pelis["description"].apply(limpiar_texto) + " "
    + df_pelis["director"].fillna('').apply(limpiar_texto) + ". "
    + df_pelis["year"].apply(lambda x: str(int(x)) if pd.notnull(x) else '') + ". "
    + df_pelis["genre"].apply(limpiar_texto) + ". "
    + df_pelis["keywords"].apply(limpiar_texto)
)
df_pelis["texto"].iloc[0]

'Herida abierta. Orin Boyd, un duro policía de una comisaría del centro de la ciudad, descubre una red de policías corruptos. Andrzej Bartkowiak. 2001. acción, crimen, suspense. vietnam war veteran, heroína, drogas, narcotraficante, corrupt cop'

## Vectorización TF-IDF

Ajustamos el `TfidfVectorizer` sobre el texto de las películas. Decisiones de configuración:

- **`stop_words`**: lista de *stopwords* en español (artículos, preposiciones, etc.). `sklearn` no trae lista en español, así que la definimos manualmente. Sin esto, palabras vacías como *de*, *la*, *que* dominarían los vectores.
- **`ngram_range=(1, 2)`**: incluimos unigramas y bigramas, para capturar expresiones como *"ciencia ficción"* o *"guerra mundial"*.
- **`min_df=2`**: ignoramos términos que aparecen en una sola película (ruido).
- **`max_df=0.5`**: ignoramos términos presentes en más del 50% del corpus (demasiado genéricos).
- **`sublinear_tf=True`**: usa `1 + log(tf)` en vez de `tf`, para que una palabra repetida muchas veces no domine.

In [6]:
STOP_ES = """a al algo algunas algunos ante antes como con contra cual cuando de del desde donde dos el ella ellas
ellos en entre era erais eran eras eres es esa esas ese eso esos esta estaba estado estais estamos estan estar este
esto estos fin fue fueron ha habia han hasta hay la las le les lo los mas me mi mis mucho muy nada ni no nos o os otra
otras otro otros para pero poco por porque que quien se sea ser si sin sobre solo son su sus tambien tan tanto te tiene
tienen toda todas todo todos tu tus un una uno unos vosotras vosotros y ya""".split()

vectorizer = TfidfVectorizer(
    stop_words=STOP_ES,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.5,
    sublinear_tf=True
)

pelis_tfidf = vectorizer.fit_transform(df_pelis["texto"].tolist())
print("Matriz TF-IDF de películas:", pelis_tfidf.shape)  # (n_pelis, vocabulario)

Matriz TF-IDF de películas: (4967, 22101)


## Representación del usuario

Cada usuario se representa con el **mismo** vectorizador, transformando un texto que concatena su **query** con el **texto de las 5 películas de su historial**. Así query e historial quedan en el mismo espacio TF-IDF que las películas.

> Igual que en el enfoque de Sentence-Transformer, esta concatenación no permite ponderar query vs. historial; pero como las queries son cortas y el historial aporta los géneros/keywords, en la práctica funciona muy bien para esta tarea.

In [7]:
def build_user_text(row, df_pelis):
    partes = [row['query']]
    for col in ['pelicula_1','pelicula_2','pelicula_3','pelicula_4','pelicula_5']:
        match = df_pelis[df_pelis['name'] == row[col]]['texto']
        if len(match):
            partes.append(match.values[0])
    return ' '.join(partes)

user_texts = usuarios.apply(lambda r: build_user_text(r, df_pelis), axis=1).tolist()
user_tfidf = vectorizer.transform(user_texts)
print("Matriz TF-IDF de usuarios:", user_tfidf.shape)

Matriz TF-IDF de usuarios: (14, 22101)


## Recomendaciones

Calculamos la **similitud coseno** entre cada usuario y todas las películas, ponemos en `-inf` las películas ya vistas (historial) y nos quedamos con el **top-5**.

In [8]:
scores = cosine_similarity(user_tfidf, pelis_tfidf)

hist_cols = ['pelicula_1', 'pelicula_2', 'pelicula_3', 'pelicula_4', 'pelicula_5']
scores_filtrado = scores.copy()
for i, row in usuarios.iterrows():
    for col in hist_cols:
        match = df_pelis[df_pelis['name'] == row[col]].index
        if len(match):
            scores_filtrado[i, match[0]] = -np.inf

# Top-5 por usuario (sobre los scores ya filtrados)
top5_indices = scores_filtrado.argsort(axis=1)[:, -5:][:, ::-1]

for i, row in usuarios.iterrows():
    print(f"\n{row['nombre']} ({row['tipo_perfil']})")
    print(f"Query: {row['query']}")
    for idx in top5_indices[i]:
        pelicula = df_pelis.iloc[idx]
        anio = int(pelicula['year']) if pd.notnull(pelicula['year']) else 'N/A'
        print(f"  {pelicula['name']} ({anio}) — {scores_filtrado[i, idx]:.4f}")


Valentina (definido)
Query: Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
  Sin City: Una dama por la que matar (2020) — 0.1786
  Misteriosa obsesión (2004) — 0.1576
  No sin mi hija (1991) — 0.1494
  Un San Valentín de muerte (2001) — 0.1437
  Lunas de hiel (1992) — 0.1307

Rodrigo (definido)
Query: Busco algo basado en hechos reales sobre corrupción o poder político
  El espía (2007) — 0.1425
  Mulholland Falls (La brigada del sombrero) (1996) — 0.1261
  Alpha Dog (2007) — 0.1153
  Suburbicón (2017) — 0.1040
  Smila: Misterio en la nieve (1997) — 0.1009

Camila (definido)
Query: Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental
  Algo en común (2005) — 0.1707
  Los padres de ella (2000) — 0.1434
  Joe contra el volcán (1990) — 0.1404
  Niñera a la fuerza (2004) — 0.1340
  Ahora los padres son ellos (2010) — 0.1317

Tomás (definido)
Query: Algo que haga pensar sobre qué es real y qué es una constr

## Evaluación

Reutilizamos la **métrica proxy por géneros** de los otros enfoques: para cada uno de los 9 perfiles definidos comparamos los géneros del top-5 contra los géneros esperados definidos a mano.

- **Recall (géneros):** proporción de géneros esperados presentes en el top-5.
- **Precision (películas):** proporción de películas del top-5 con al menos un género esperado.
- **F1:** media armónica.

In [9]:
etiquetas_a_ojo_def = [
    ["suspense", "terror", "drama"],
    ["crimen", "biografía", "historia"],
    ["comedia", "romance", "drama"],
    ["acción", "ciencia ficción", "suspense"],
    ["animación", "drama", "aventura"],
    ["crimen", "acción", "comedia"],
    ["música", "drama", "comedia"],
    ["acción", "crimen", "aventura"],
    ["drama", "romance", "comedia"],
]

In [10]:
resultados_eval = []

for user_idx, row in usuarios.head(9).iterrows():
    print(f"\n{'='*70}")
    print(f"{row['nombre']} ({row['tipo_perfil']})")
    print(f"Query: {row['query']}")
    generos_esperados = set(etiquetas_a_ojo_def[user_idx])
    print(f"Géneros Esperados: {', '.join(generos_esperados)}")
    print(f"{'='*70}")

    generos_recomendados = Counter()
    peliculas_buenas = 0
    peliculas_malas = []

    print("\nTop-5 Recomendaciones:")
    for rank, idx in enumerate(top5_indices[user_idx], 1):
        pelicula = df_pelis.iloc[idx]
        score = scores[user_idx, idx]

        generos_list = [g.strip() for g in pelicula['genre'].strip('[]').split(',')]
        generos_pelicula = set(generos_list)
        generos_recomendados.update(generos_list)

        es_buena = bool(generos_pelicula & generos_esperados)
        if es_buena:
            peliculas_buenas += 1
            marker = "✓"
        else:
            peliculas_malas.append(pelicula['name'])
            marker = "✗"

        anio = int(pelicula['year']) if pd.notnull(pelicula['year']) else 'N/A'
        print(f"  {rank}. [{marker}] {pelicula['name']} ({anio}) — {score:.4f}")
        print(f"     Géneros: {', '.join(generos_list)}")

    generos_capturados = set(generos_recomendados.keys())
    recall = len(generos_capturados & generos_esperados) / len(generos_esperados)
    precision = peliculas_buenas / 5
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\nMÉTRICAS:")
    print(f"  Recall (géneros):      {recall:.1%}  ({len(generos_capturados & generos_esperados)}/{len(generos_esperados)})")
    print(f"  Precision (películas): {precision:.1%}  ({peliculas_buenas}/5)")
    print(f"  F1-Score:              {f1:.1%}")

    if peliculas_malas:
        print(f"\nPelículas problemáticas (sin géneros esperados):")
        for pelicula in peliculas_malas:
            print(f"    - {pelicula}")

    resultados_eval.append({
        'Usuario': row['nombre'],
        'Recall': recall,
        'Precision': precision,
        'F1': f1,
        'Películas Malas': len(peliculas_malas)
    })

print(f"\n\n{'='*70}")
print("RESUMEN DE EVALUACIÓN — TF-IDF")
print(f"{'='*70}")
df_eval = pd.DataFrame(resultados_eval)
df_eval.to_csv("evaluacion_tfidf.csv", index=False)
print(df_eval.to_string(index=False))
print(f"\nPromedios:")
print(f"  Recall:    {df_eval['Recall'].mean():.1%}")
print(f"  Precision: {df_eval['Precision'].mean():.1%}")
print(f"  F1-Score:  {df_eval['F1'].mean():.1%}")


Valentina (definido)
Query: Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Géneros Esperados: drama, suspense, terror

Top-5 Recomendaciones:
  1. [✓] Sin City: Una dama por la que matar (2020) — 0.1786
     Géneros: acción, crimen, suspense
  2. [✓] Misteriosa obsesión (2004) — 0.1576
     Géneros: drama, misterio, ciencia ficción
  3. [✓] No sin mi hija (1991) — 0.1494
     Géneros: drama, suspense
  4. [✓] Un San Valentín de muerte (2001) — 0.1437
     Géneros: terror, misterio, suspense
  5. [✓] Lunas de hiel (1992) — 0.1307
     Géneros: drama, romance, suspense

MÉTRICAS:
  Recall (géneros):      100.0%  (3/3)
  Precision (películas): 100.0%  (5/5)
  F1-Score:              100.0%

Rodrigo (definido)
Query: Busco algo basado en hechos reales sobre corrupción o poder político
Géneros Esperados: historia, crimen, biografía

Top-5 Recomendaciones:
  1. [✓] El espía (2007) — 0.1425
     Géneros: biografía, crimen, drama
  2. [✓] Mulhol

## Análisis y comparación

**TF-IDF resultó el enfoque más fuerte en la métrica por géneros.** La razón es directa: el texto de usuario incluye los géneros y keywords de su historial, y TF-IDF los matchea de forma **literal** contra los géneros/keywords de las películas candidatas. Donde los embeddings se "desviaban" hacia el tono semántico (p. ej. en Lucía, que pedía *animación* pero recibía dramas existenciales), TF-IDF ancla la recomendación en la coincidencia exacta de la palabra *animación*.

**Pero ese mismo mecanismo es su límite.** TF-IDF solo ve palabras, no significado:

- Si una query pide *"una amenaza invisible"* y la sinopsis relevante dice *"un acosador desconocido"*, no hay coincidencia léxica y TF-IDF no las relaciona; un embedding sí.
- Es sensible al vocabulario exacto, sinónimos y al *spanglish* del corpus.
- Parte del excelente resultado se explica porque **la métrica de evaluación también es por género**: TF-IDF y la métrica comparten la misma señal léxica (los géneros), lo que la favorece. Con una evaluación basada en relevancia semántica real, la ventaja se reduciría.

**Conclusión:** TF-IDF es un baseline content-based muy competitivo y barato (sin GPU ni modelos preentrenados), ideal cuando la query y el contenido comparten vocabulario. Los embeddings siguen siendo preferibles cuando importa el significado más allá de las palabras exactas.

## Variante: ponderación query / historial (estilo Enfoque 2)

Hasta acá concatenamos query + historial en un solo texto, sin controlar cuánto pesa cada parte. Replicamos ahora la idea del **Enfoque 2 (embeddings + LLM)**: calculamos por separado el vector TF-IDF de la **query** y el del **historial** (promedio de las 5 películas vistas), y los combinamos con un **promedio ponderado** según la **intención de la query**.

Las categorías provienen de la clasificación con **Ollama (llama3.2)** del Enfoque 2; las fijamos acá para que este notebook sea reproducible sin levantar el LLM:

- **`normal`** (query bien definida) → más peso a la query `[0.7, 0.3]`, top-5 más similares.
- **`historial_positivo`** (query vaga / "lo de siempre") → más peso al historial `[0.1, 0.9]`, top-5 más similares.
- **`historial_negativo`** (pide algo distinto) → más peso al historial `[0.1, 0.9]`, pero se eligen las **menos** similares (bottom-5).

In [11]:
# Vector TF-IDF de la query y del historial, por separado
query_tfidf = vectorizer.transform(usuarios['query'].tolist()).toarray()

# Historial: promedio de los vectores TF-IDF de las 5 películas vistas
name_to_idx = {name: i for i, name in enumerate(df_pelis['name'])}
hist_tfidf = []
for _, row in usuarios.iterrows():
    idxs = [name_to_idx[row[c]] for c in hist_cols if row[c] in name_to_idx]
    hist_tfidf.append(np.asarray(pelis_tfidf[idxs].mean(axis=0)).ravel())
hist_tfidf = np.vstack(hist_tfidf)

print("query_tfidf:", query_tfidf.shape, "| hist_tfidf:", hist_tfidf.shape)

query_tfidf: (14, 22101) | hist_tfidf: (14, 22101)


In [12]:
# Parámetros por categoría (idénticos al Enfoque 2)
CATEGORIA_PARAMS = {
    'normal':             {'weights': [0.7, 0.3], 'direction': 'top-5'},
    'historial_positivo': {'weights': [0.1, 0.9], 'direction': 'top-5'},
    'historial_negativo': {'weights': [0.1, 0.9], 'direction': 'bottom-5'},
}

# Categoría de cada usuario, obtenida con Ollama (llama3.2) en el Enfoque 2
categorias_usuarios = [
    'normal',              # U01 Valentina
    'historial_positivo',  # U02 Rodrigo
    'normal',              # U03 Camila
    'historial_positivo',  # U04 Tomás
    'normal',              # U05 Lucía
    'normal',              # U06 Martín
    'normal',              # U07 Sofía
    'normal',              # U08 Diego
    'normal',              # U09 Elena
    'normal',              # U10
    'normal',              # U11
    'historial_negativo',  # U12
    'normal',              # U13
    'historial_positivo',  # U14
]

In [13]:
# Vector de usuario = promedio ponderado de query e historial
user_vec_pond = []
for i, cat in enumerate(categorias_usuarios):
    wq, wh = CATEGORIA_PARAMS[cat]['weights']
    user_vec_pond.append(wq * query_tfidf[i] + wh * hist_tfidf[i])
user_vec_pond = np.vstack(user_vec_pond)

scores_pond = cosine_similarity(user_vec_pond, pelis_tfidf)

# Filtrar historial
scores_pond_filtrado = scores_pond.copy()
for i, row in usuarios.iterrows():
    for col in hist_cols:
        match = df_pelis[df_pelis['name'] == row[col]].index
        if len(match):
            scores_pond_filtrado[i, match[0]] = -np.inf

# Top-5 por usuario según dirección (bottom-5 para historial_negativo)
top5_pond = []
for i, cat in enumerate(categorias_usuarios):
    if CATEGORIA_PARAMS[cat]['direction'] == 'bottom-5':
        sv = scores_pond_filtrado[i].copy()
        sv[sv == -np.inf] = np.inf  # que el historial no caiga en el bottom
        top5_pond.append(sv.argsort()[:5])
    else:
        top5_pond.append(scores_pond_filtrado[i].argsort()[-5:][::-1])
top5_pond = np.array(top5_pond)

In [14]:
resultados_pond = []

for user_idx, row in usuarios.head(9).iterrows():
    cat = categorias_usuarios[user_idx]
    print(f"\n{'='*70}")
    print(f"{row['nombre']} ({row['tipo_perfil']}) | Categoría: {cat} | Pesos: {CATEGORIA_PARAMS[cat]['weights']}")
    print(f"Query: {row['query']}")
    generos_esperados = set(etiquetas_a_ojo_def[user_idx])
    print(f"Géneros Esperados: {', '.join(generos_esperados)}")
    print(f"{'='*70}")

    generos_recomendados = Counter()
    peliculas_buenas = 0
    peliculas_malas = []

    print("\nTop-5 Recomendaciones:")
    for rank, idx in enumerate(top5_pond[user_idx], 1):
        pelicula = df_pelis.iloc[idx]
        score = scores_pond[user_idx, idx]
        generos_list = [g.strip() for g in pelicula['genre'].strip('[]').split(',')]
        generos_recomendados.update(generos_list)
        es_buena = bool(set(generos_list) & generos_esperados)
        if es_buena:
            peliculas_buenas += 1; marker = "✓"
        else:
            peliculas_malas.append(pelicula['name']); marker = "✗"
        anio = int(pelicula['year']) if pd.notnull(pelicula['year']) else 'N/A'
        print(f"  {rank}. [{marker}] {pelicula['name']} ({anio}) — {score:.4f}")
        print(f"     Géneros: {', '.join(generos_list)}")

    generos_capturados = set(generos_recomendados.keys())
    recall = len(generos_capturados & generos_esperados) / len(generos_esperados)
    precision = peliculas_buenas / 5
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    print(f"\nMÉTRICAS: Recall {recall:.1%} | Precision {precision:.1%} | F1 {f1:.1%}")

    resultados_pond.append({'Usuario': row['nombre'], 'Categoría': cat,
                            'Recall': recall, 'Precision': precision, 'F1': f1})

print(f"\n\n{'='*70}")
print("RESUMEN — TF-IDF CON PONDERACIÓN QUERY/HISTORIAL")
print(f"{'='*70}")
df_pond = pd.DataFrame(resultados_pond)
df_pond.to_csv("evaluacion_tfidf_ponderado.csv", index=False)
print(df_pond.to_string(index=False))
print(f"\nPromedios:")
print(f"  Recall:    {df_pond['Recall'].mean():.1%}")
print(f"  Precision: {df_pond['Precision'].mean():.1%}")
print(f"  F1-Score:  {df_pond['F1'].mean():.1%}")


Valentina (definido) | Categoría: normal | Pesos: [0.7, 0.3]
Query: Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Géneros Esperados: drama, suspense, terror

Top-5 Recomendaciones:
  1. [✓] Cabin fever 2 (2011) — 0.1393
     Géneros: terror
  2. [✗] Los ríos de color púrpura 2: Los ángeles del apocalipsis (2004) — 0.1369
     Géneros: acción, crimen, misterio
  3. [✓] Inseparables (1989) — 0.1356
     Géneros: drama, terror, suspense
  4. [✓] Suavemente me mata (2002) — 0.1240
     Géneros: drama, misterio, romance
  5. [✓] El hombre sin sombra (2000) — 0.1154
     Géneros: acción, terror, ciencia ficción

MÉTRICAS: Recall 100.0% | Precision 80.0% | F1 88.9%

Rodrigo (definido) | Categoría: historial_positivo | Pesos: [0.1, 0.9]
Query: Busco algo basado en hechos reales sobre corrupción o poder político
Géneros Esperados: historia, crimen, biografía

Top-5 Recomendaciones:
  1. [✓] El espía (2007) — 0.1479
     Géneros: biografía, crim

## Comparación: concatenación simple vs. ponderación query/historial

| Variante TF-IDF | Recall | Precision | F1 |
|---|---|---|---|
| Concatenación simple (query + historial en un texto) | 96.3% | 97.8% | **96.5%** |
| Ponderación query/historial (estilo Enfoque 2) | 92.6% | 91.1% | 91.0% |

A diferencia de lo que ocurría con los embeddings densos, en TF-IDF la ponderación explícita **no mejora** —incluso baja un poco— respecto de la simple concatenación. La razón es la naturaleza dispersa de TF-IDF:

- La query es muy corta (mediana ~15 palabras) y muchos de sus términos abstractos (*"amenaza invisible"*, *"reconectar"*) no existen en el vocabulario del corpus o aparecen en pocas películas. Su vector TF-IDF es por lo tanto **muy ralo**.
- Al darle peso 0.7 a ese vector ralo, unos pocos términos pasan a dominar la similitud, lo que vuelve la recomendación más errática (p. ej. Camila baja de F1 100% a 73%).
- En la concatenación simple, en cambio, los géneros y keywords del historial aportan la mayor parte de la masa léxica y estabilizan la recomendación.

**Conclusión:** la ponderación query/historial es valiosa cuando trabajamos con embeddings densos (donde la query aporta una señal semántica rica aunque sea corta), pero con TF-IDF la concatenación directa es preferible. El mecanismo de control que ayuda en un espacio de representación puede perjudicar en otro: la decisión de cómo combinar las señales **depende de la representación elegida**.